In [ ]:
# Hướng dẫn viết code và commit ngay trên local và chạy trên hạ tầng GCP

# Lựa chọn interpreter là jupyter server và điền URL
# http://<external_master_ip>:8888/tree?token=<token_>
# Do không cấu hình bảo mật kỹ nên không chia sẻ URL ra ngoài
# Nếu kết nối thành công với jupyter server của node master trên GCP, có thể dùng các lệnh với dấu ! như bên dưới để cài python lib qua pip lên master, đối với các worker thì: !ssh hadoop-worker-1 "/home/tiennd3886/BTL-MMDS/env/bin/python3 -m pip install -U <ten_goi_can_cai>"
# Các data raw và chia train val test đã đưa lên hadoop, xem bên dưới để biết thêm đường dẫn để sử dụng.

# cluster trên GCP: 3 máy E2 Standar, 4 vcpu và 16Gb Ram mỗi máy, master có data node và cũng làm worker, 
# tên master node: hadoop-master, 
# 2 worker: hadoop-worker-1, hadoop-worker-2

# Các file sinh ra trong quá trình code, kể cả model hãy lưu trên hadoop tại thư mục tương ứng: 
# user/<ten_cua_ban>/
# Sau đó có thể tải model về local thông qua Web quản lý của Hadoop: http://<external_master_ip>:9870/explorer.html#/user, 
# nhưng có thể sẽ bị chuyển hướng sang http://hadoop-master:9864/webhdfs/v1... hoặc http://hadoop-worker-1:9864/webhdfs/v1...
# thì cần thay cụm hadooop-master, hadoop-worker-1 , 2 thành external ip tương ứng để tải được

# Nếu code lỗi chỉ cần restart kernel để xóa cache, giải phóng ram
# Các trường hợp restart kenel không giải quyết được:
## Nếu code đã ngừng chạy mà nghi ngờ có tiến trình Spark bị treo, kiểm tra bằng: 
##!yarn application -list
## và kill tiến trình đó: !yarn application -kill <application_id>

## Nếu dùng tính năng checkpoint() của HDFS khi huyến luyện model thì khi code lỗi cần xóa checkpoint để giải phóng dung lượng:
## !hdfs dfs -rm -r -skipTrash /đường_dẫn_thư_mục_checkpoint

## Rác sinh ra trong quá trình Shuffle, khi thấy ổ cứng báo đầy hoặc Spark báo lỗi không đủ dung lượng đĩa cục bộ, lệnh sau để dọn sạch rác Spark:
## !sudo rm -rf /tmp/spark-*
## !sudo rm -rf /tmp/blockmgr-*

# Cảnh báo để không làm sập máy Master lần nữa
# Không dùng df.toPandas() trên toàn bộ dataframe: Chỉ dùng .toPandas() sau khi đã tổng hợp hoặc giới hạn dòng (df.limit(100).toPandas()).
# Hạn chế dùng df.collect(): Tương tự như trên, lệnh này kéo mảng dữ liệu về Driver. 
# Hãy thay thế bằng df.show() nếu chỉ muốn xem trước dữ liệu, hoặc lưu thẳng xuống HDFS bằng df.write.parquet(...).

In [ ]:
# ví dụ cài thư viện python trên master:
!pip install numpy
# ví dụ cài thư viện python trên worker:
!ssh hadoop-worker-1 "/home/tiennd3886/BTL-MMDS/env/bin/python3 -m pip install -U numpy"

In [ ]:
!python3 --version
!which python3
!pip --version
!hdfs dfs -df -h
!hdfs dfsadmin -report
!yarn node -list
!hdfs dfs -mkdir -p /user/test1
!hdfs dfs -rm -r -skipTrash /user/test1

Python 3.12.3
/home/tiennd3886/BTL-MMDS/env/bin/python3
pip 26.1.1 from /home/tiennd3886/BTL-MMDS/env/lib/python3.12/site-packages/pip (python 3.12)
Filesystem                    Size    Used  Available  Use%
hdfs://hadoop-master:9000  239.0 G  29.2 G    150.9 G   12%
Configured Capacity: 256660299776 (239.03 GB)
Present Capacity: 193421707598 (180.14 GB)
DFS Remaining: 162080192846 (150.95 GB)
DFS Used: 31341514752 (29.19 GB)
DFS Used%: 16.20%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (3):

Name: 10.128.0.2:9866 (hado

In [ ]:
!hdfs dfs -rm -r -skipTrash /user/test1
print("--- list dir on hdfs:")
!hdfs dfs -ls
print("--- list dir /user on hdfs:")
!hdfs dfs -ls /user
print("--- make dir /user/test1 on hdfs and list dir to check:")
!hdfs dfs -mkdir -p /user/test1
!hdfs dfs -ls /user
print("--- remove dir /user/test1 on hdfs:")
!hdfs dfs -rm -r -skipTrash /user/test1
!hdfs dfs -mkdir -p /user/test1

print("--- put file to hdfs:")
## thư mục /tmp là thư mục nằm ở root của máy master, không phải thư mục trên máy local của bạn
# %%bash
!echo \"hello hdfs\" > /tmp/hdfs_demo.txt
!hdfs dfs -put -f /tmp/hdfs_demo.txt /user/test1
!hdfs dfs -ls /user/test1

print("--- copy file on hdfs:")
!hdfs dfs -cp /user/test1/hdfs_demo.txt /user/test1/hdfs_demo_copy.txt
!hdfs dfs -ls /user/test1
!hdfs dfs -cat /user/test1/hdfs_demo.txt

print("--- Get file tu HDFS ve local:")
!hdfs dfs -get /user/test1/hdfs_demo.txt /tmp/hdfs_demo_from_hdfs.txt

print("--- delete file on hdfs:")
!hdfs dfs -rm /user/test1/hdfs_demo_copy.txt

Deleted /user/test1
--- list dir on hdfs:
Found 4 items
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 14:03 feature_engineering
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 14:38 models
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 16:13 results
drwxrwxrwx   - tiennd3886 supergroup          0 2026-05-25 10:18 spark-logs
--- list dir /user on hdfs:
Found 3 items
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:31 /user/data
drwxrwxrwt   - tiennd     supergroup          0 2026-05-25 06:41 /user/spark
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 14:38 /user/tiennd3886
--- make dir /user/test1 on hdfs and list dir to check:
Found 4 items
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:31 /user/data
drwxrwxrwt   - tiennd     supergroup          0 2026-05-25 06:41 /user/spark
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 17:07 /user/test1
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 14

In [13]:
print("--- thư mục dự án trên master:")
!ls
print("--- thư mục user hiện tại trên master:")
!ls ..

--- thư mục dự án trên master:
README.md  data  gcp_config	      requirements.txt
code	   env	 real_cluster_config  virtual_cluster_config
--- thư mục user hiện tại trên master:
BTL-MMDS  hadoop-3.3.6	snap  spark-3.5.1-bin-hadoop3


In [14]:
print("--- data raw đã được đưa lên hdfs tại /user/data/raw:")
!hdfs dfs -ls /user/data/raw

--- data raw đã được đưa lên hdfs tại /user/data/raw:
Found 121 items
-rw-r--r--   2 tiennd3886 supergroup      12065 2026-05-25 08:03 /user/data/raw/taxi_zone_lookup.csv
-rw-r--r--   2 tiennd3886 supergroup  151251087 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-01.parquet
-rw-r--r--   2 tiennd3886 supergroup  158113739 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-02.parquet
-rw-r--r--   2 tiennd3886 supergroup  170019864 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-03.parquet
-rw-r--r--   2 tiennd3886 supergroup  165552992 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-04.parquet
-rw-r--r--   2 tiennd3886 supergroup  165807271 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-05.parquet
-rw-r--r--   2 tiennd3886 supergroup  156288749 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-06.parquet
-rw-r--r--   2 tiennd3886 supergroup  144621113 2026-05-25 12:28 /user/data/raw/yellow_tripdata_2016-07.parquet
-rw-r--r--   2 tiennd3886 supergroup  1398757

In [16]:
print("--- data raw đã được chia thành các tập train, test, val trên hdfs tại /user/data/train /user/data/val /user/data/test:")
!hdfs dfs -ls /user/data/
print("--- các tập được chia như trong file eda/data_split.ipynb (xem output cell)")

--- data raw đã được chia thành các tập train, test, val trên hdfs tại /user/data/train /user/data/val /user/data/test:
Found 5 items
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 08:02 /user/data/preprocess
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:29 /user/data/raw
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:39 /user/data/test
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:36 /user/data/train
drwxr-xr-x   - tiennd3886 supergroup          0 2026-05-25 12:37 /user/data/val
--- các tập được chia như trong file eda/data_split.ipynb (xem output cell)


# Cell cau hinh Spark session khuyen khich nen su dung theo:

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

# Change username
BASE_HDFS = "/user/your_name_or_your_papers_here_"
RAW_TRAIN = "/user/data/train/*.parquet"
RAW_VAL = "/user/data/val/*.parquet"
RAW_TEST = "/user/data/test/*.parquet"

OUT_DENSE = f"{BASE_HDFS}/feature_engineering/demand_prediction_dense_30m"
OUT_FEATURES = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m"

BIN_SECONDS = 1800
BIN_COL = "pickup_bin_30m"
ZONE_COL = "PULocationID"
TARGET_COL = "pickup_demand_t1"
CURRENT_COL = "pickup_demand"

spark = (
    SparkSession.builder
    .appName("DemandPredictionFeatureEngineering_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "3") 
    .config("spark.executor.cores", "3") 
    .config("spark.executor.memory", "8g") 
    .config("spark.executor.memoryOverhead", "2g") 
    .config("spark.driver.memory", "6g") 
    .config("spark.driver.memoryOverhead", "2g") 
    .config("spark.sql.shuffle.partitions", "18")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    # .config("spark.sql.execution.arrow.pyspark.enabled", "false")  # DISABLE Arrow backend to avoid ChunkedArray
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 16:48:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 16:48:40 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [ ]:
# PICKUP_COL_CANDIDATES = ["tpep_pickup_datetime", "pickup_datetime"]

# from functools import reduce

# def load_split(path_glob: str, split_name: str):
#     sc = spark.sparkContext
#     fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
#     FileStatus_arr = fs.globStatus(sc._jvm.org.apache.hadoop.fs.Path(path_glob))
#     if not FileStatus_arr:
#         raise ValueError(f"No files found for {path_glob}")
#     file_paths = [status.getPath().toString() for status in FileStatus_arr]
    
#     dfs = []
#     for p in file_paths:
#         df = spark.read.parquet(p)
#         pickup_col = next((c for c in PICKUP_COL_CANDIDATES if c in df.columns), None)
#         if pickup_col is None or ZONE_COL not in df.columns:
#             print(f"Skipping {p} due to missing columns")
#             continue
#         cleaned = (
#             df.select(
#                 F.to_timestamp(F.col(pickup_col)).alias("pickup_ts"),
#                 F.col(ZONE_COL).cast("int").alias(ZONE_COL),
#             )
#             .where(F.col("pickup_ts").isNotNull())
#             .where(F.col(ZONE_COL).isNotNull())
#             .where(F.col(ZONE_COL) > 0)
#             .withColumn("split", F.lit(split_name))
#         )
#         cleaned = cleaned.where(F.year("pickup_ts").between(2020, 2025))
#         cleaned = cleaned.where(~((F.year("pickup_ts") == 2020) & (F.month("pickup_ts").between(3, 6))))
#         dfs.append(cleaned)
        
#     if not dfs:
#         raise ValueError(f"No valid data to union for {split_name}")
    
#     return reduce(lambda a, b: a.unionByName(b), dfs)

# train_df = load_split(RAW_TRAIN, "train")
# val_df = load_split(RAW_VAL, "val")
# test_df = load_split(RAW_TEST, "test")
# raw_df = train_df.unionByName(val_df).unionByName(test_df).cache()
# print("Raw rows:", raw_df.count())


In [ ]:
spark.catalog.clearCache()
spark.stop()